## Notebook Purpose: To generate evaluation tables for how mnay true/false positive/negatives there were for varying detection thresholds and SNR thresholds applied on the bd2  detections post-overlap removal with threshold 12ms and using the TP-classification threshold of 12ms

## Imports Section:

In [1]:
import numpy as np
import pandas as pd
import random
import scipy
from scipy import stats
import datetime as dt
import dask.dataframe as dd

In [2]:
import glob
import matplotlib.pyplot as plt
from matplotlib import colors
import soundfile as sf
import matplotlib.patches as patches
from pathlib import Path

In [3]:
import convert_between_detector_file_formats as conversion
import generate_precision_recall_evaluation_table as evaltable

In [4]:
LABEL_FOR_GROUPS = {
                    0: 'LF', 
                    1: 'HF'
                    }

## Function and Constant Definitions

In [5]:
FREQ_COLORS = {'LF':'cyan', 'HF':'orange'}

In [6]:
FREQ_COLORS = {'LF':'cyan', 'HF':'orange'}
FILE_SITES = {'20220730_053000':'Carp',
 '20220826_070000':'Central',
 '20220727_083000':'Foliage',
 '20220829_090000':'Foliage'}

RAVENPRO_TABLE_EXTENSION = ".txt"
RAVENPRO_TABLE_FORMAT = "\t"
RAVENTXT_HUMAN_ANNOTATIONS_SAVE_DIR = f'{Path.home()}/Documents/Research/mila_files/mila-human-wav-txt'
BD2_DETS_SAVE_DIR = Path(f'20250130__group_threshold_sweep_results')
TP_CLASSIFICATION_THRESHOLD = 0.012
OVERLAP_TIME_THRESHOLD = 0.012
SITE_NAMES = {'Carp':'Carp Pond', 'Foliage':'Foliage', 'Central':'Central Pond'}

In [7]:
test = float('-inf')

In [8]:
import math
math.isinf(test)

True

In [9]:
file_keys = list(FILE_SITES.keys())
file_keys

['20220730_053000', '20220826_070000', '20220727_083000', '20220829_090000']

## Following the confusion matrix structure shown below

![image](example_confusion_matrix.png)

In [10]:
batdetect2_eval = pd.DataFrame()
batdetect2_eval_LF = pd.DataFrame()
batdetect2_eval_HF = pd.DataFrame()
for file_key in file_keys:
    wav_filename = file_key
    site = FILE_SITES[file_key]
    plot_file = Path(f'{RAVENTXT_HUMAN_ANNOTATIONS_SAVE_DIR}/{wav_filename}.WAV')
    raventxt_human_filename = f'{wav_filename}_manually_verified_by_AK_with_groups'

    snr_included_ravenpro_human_txt = pd.read_csv(f'{RAVENTXT_HUMAN_ANNOTATIONS_SAVE_DIR}/{raventxt_human_filename}_ravenpro_SNR_aditya_SNR_and_peakfreqtime.txt', 
                                                  sep=RAVENPRO_TABLE_FORMAT)
    bd2_human_df_file = snr_included_ravenpro_human_txt.copy()
    bd2_human_df_file = bd2_human_df_file.drop(columns=['Selection', 'View', 'Channel'])
    bd2_human_df_file.rename(columns={'Begin Time (s)':'start_time',
                                'End Time (s)':'end_time',
                                'Low Freq (Hz)':'low_freq',
                                'High Freq (Hz)':'high_freq',
                                'Delta Time (s)':'delta_time_s',
                                'Manually-Verified Phonic Group':'freq_group',
                                'SNR NIST Quick (dB)':'snr_nist_quick_dB'}, inplace=True)
    bd2_human_df_file.sort_values('start_time', inplace=True)

    args = dict()
    args['chunk_size'] = 2
    args['detection_threshold'] = 0.00
    ones = int(args['detection_threshold'])
    decimals = int(int(100*(args['detection_threshold'])) % 100)
    threshold_tag = f"threshold{ones}p{decimals:02}"
    save_loc_reduced_overlaps = Path(f"bd2__{threshold_tag}_chunksize{int(args['chunk_size'])}_{wav_filename}_REDUCED_OVERLAPS.csv")
    filepath_reduced_overlaps = BD2_DETS_SAVE_DIR / save_loc_reduced_overlaps
    bd2_file_all_dets = pd.read_csv(filepath_reduced_overlaps, sep=',', index_col=0)
    bd2_file_all_dets.rename(columns={'KMEANS_CLASSES':'freq_group'}, inplace=True)

    bd2_human_df_file_LF = bd2_human_df_file[bd2_human_df_file['freq_group']=='LF'].copy()
    batdetect2_df_file_LF_reduced_overlaps = bd2_file_all_dets[bd2_file_all_dets['freq_group']=='LF'].copy()
    batdetect2_eval_file_LF = evaltable.generate_evaluation_df(bd2_human_df_file_LF, batdetect2_df_file_LF_reduced_overlaps, TP_CLASSIFICATION_THRESHOLD)
    batdetect2_eval_file_LF['input_file_name'] = [f'{wav_filename}.WAV']*len(batdetect2_eval_file_LF)
    batdetect2_eval_file_LF['site_name'] = [SITE_NAMES[site]]*len(batdetect2_eval_file_LF)

    bd2_human_df_file_HF = bd2_human_df_file[bd2_human_df_file['freq_group']=='HF'].copy()
    batdetect2_df_file_HF_reduced_overlaps = bd2_file_all_dets[bd2_file_all_dets['freq_group']=='HF'].copy()
    batdetect2_eval_file_HF = evaltable.generate_evaluation_df(bd2_human_df_file_HF, batdetect2_df_file_HF_reduced_overlaps, TP_CLASSIFICATION_THRESHOLD)
    batdetect2_eval_file_HF['input_file_name'] = [f'{wav_filename}.WAV']*len(batdetect2_eval_file_HF)
    batdetect2_eval_file_HF['site_name'] = [SITE_NAMES[site]]*len(batdetect2_eval_file_HF)

    batdetect2_eval_LF = pd.concat([batdetect2_eval_LF, batdetect2_eval_file_LF])
    batdetect2_eval_HF = pd.concat([batdetect2_eval_HF, batdetect2_eval_file_HF])

0.0 0.01
0.0 0.02
0.0 0.03
0.0 0.04
0.0 0.05
0.0 0.06
0.0 0.07
0.0 0.08
0.0 0.09
0.0 0.1
0.0 0.11
0.0 0.12
0.0 0.13
0.0 0.14
0.0 0.15
0.0 0.16
0.0 0.17
0.0 0.18
0.0 0.19
0.0 0.2
0.0 0.21
0.0 0.22
0.0 0.23
0.0 0.24
0.0 0.25
0.0 0.26
0.0 0.27
0.0 0.28
0.0 0.29
0.0 0.3
0.0 0.31
0.0 0.32
0.0 0.33
0.0 0.34
0.0 0.35
0.0 0.36
0.0 0.37
0.0 0.38
0.0 0.39
0.0 0.4
0.0 0.41
0.0 0.42
0.0 0.43
0.0 0.44
0.0 0.45
0.0 0.46
0.0 0.47
0.0 0.48
0.0 0.49
0.0 0.5
0.0 0.51
0.0 0.52
0.0 0.53
0.0 0.54
0.0 0.55
0.0 0.56
0.0 0.57
0.0 0.58
0.0 0.59
0.0 0.6
1.0 0.01
1.0 0.02
1.0 0.03
1.0 0.04
1.0 0.05
1.0 0.06
1.0 0.07
1.0 0.08
1.0 0.09
1.0 0.1
1.0 0.11
1.0 0.12
1.0 0.13
1.0 0.14
1.0 0.15
1.0 0.16
1.0 0.17
1.0 0.18
1.0 0.19
1.0 0.2
1.0 0.21
1.0 0.22
1.0 0.23
1.0 0.24
1.0 0.25
1.0 0.26
1.0 0.27
1.0 0.28
1.0 0.29
1.0 0.3
1.0 0.31
1.0 0.32
1.0 0.33
1.0 0.34
1.0 0.35
1.0 0.36
1.0 0.37
1.0 0.38
1.0 0.39
1.0 0.4
1.0 0.41
1.0 0.42
1.0 0.43
1.0 0.44
1.0 0.45
1.0 0.46
1.0 0.47
1.0 0.48
1.0 0.49
1.0 0.5
1.0 0.51
1.0 0.52
1.0

In [12]:
batdetect2_eval.to_csv(BD2_DETS_SAVE_DIR / '20250130__bd2_eval_results_per_thresh_and_file.csv')
batdetect2_eval_LF.to_csv(BD2_DETS_SAVE_DIR / '20250130__bd2_LFeval_results_per_thresh_and_file.csv')
batdetect2_eval_HF.to_csv(BD2_DETS_SAVE_DIR / '20250130__bd2_HFeval_results_per_thresh_and_file.csv')